In [37]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

**Scrapped Stock Name and Change Only**

In [ ]:
# URL to scrape
base_url = "https://www.stocktitan.net"
url = base_url+"/scanner/momentum"

columns = ['name','change', 'source']
df = pd.DataFrame(columns=columns)

# Send a GET request to the URL
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
response = requests.get(url, headers=headers)

# Check if request was successful
if response.status_code != 200:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")
    exit()

# Parse the HTML content using BeautifulSoup
soup = BeautifulSoup(response.content, "html.parser")


top_gainers_div = soup.find("div", {"id": 'gainers'})
top_gainers_table = top_gainers_div.find("div", {"class": 'body'}).find_all("div",{'class':'content'})              #.find_all("div",{'class':['feed-row', 'feed-border-gradient', 'rounded mb-3']})

print(len(top_gainers_table))

for i in range(6):
    print("I -> ",i)
    name_news_div = top_gainers_table[i].find("div", {'class': 'symbol'})

    stock = name_news_div.get_text().split(':')[0].strip() if name_news_div.get_text() else None
    # news_info_link = name_news_div.find('a').get('href') if name_news_div.find('a').get('href') else None
    

    card_info_divs = top_gainers_table[i].find("div", {'class': "data-group"})
    change = card_info_divs.find('span', {'class': 'price-change-ratio'}).get_text().replace(r"\r\n",'')
    change_num = float(re.sub(r'[+%]', '', change))


    print('stock_name: ',stock)
    print('change: ',change_num)

    new_row = {
    'name': stock,
    'change': change_num,
    'source': "StockTitan"
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv("stocktitan_name_change.csv")



25
I ->  0
stock_name:  SPHL
change:  72.0
I ->  1
stock_name:  KIDZ
change:  52.0
I ->  2
stock_name:  ADIL
change:  42.0
I ->  3
stock_name:  SOPA
change:  38.0
I ->  4
stock_name:  CNEY
change:  31.0
I ->  5
stock_name:  PTHL
change:  23.0


/tmp/ipykernel_691633/3137338289.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


In [ ]:
import re

text = "+47.00%"
cleaned = re.sub(r'[+%]', '', text)
number = float(cleaned)
print(number) 

47.0
<class 'str'>


In [32]:
# URL to scrape
base_url = "https://www.stocktitan.net"
url = base_url+"/scanner/momentum"

columns = ['name', 'market_cap', 'change','float', 'volume', 'short_percent', 'industry', 'sector', 'country', 'news_link','news_impact_star', 'news_sentiment_star']
df = pd.DataFrame(columns=columns)

# Send a GET request to the URL
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
response = requests.get(url, headers=headers)

# Check if request was successful
if response.status_code != 200:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")
    exit()

# Parse the HTML content using BeautifulSoup
soup = BeautifulSoup(response.content, "html.parser")


top_gainers_div = soup.find("div", {"id": 'gainers'})
top_gainers_table = top_gainers_div.find("div", {"class": 'body'}).find_all("div",{'class':'content'})              #.find_all("div",{'class':['feed-row', 'feed-border-gradient', 'rounded mb-3']})

print(len(top_gainers_table))

for i in range(4):
    print("I -> ",i)
    name_news_div = top_gainers_table[i].find("div", {'class': 'symbol'})

    stock_name = name_news_div.get_text().split(':')[0].strip() if name_news_div.get_text() else None
    news_info_link = name_news_div.find('a').get('href') if name_news_div.find('a').get('href') else None
    
    print('stock_name: ',stock_name)
    print('news_info_link: ',news_info_link)

    card_info_divs = top_gainers_table[i].find("div", {'class': "data-group"})
    change_value = card_info_divs.find('span', {'class': 'price-change-ratio'}).get_text().replace(r"\r\n",'')
    market_cap_value = card_info_divs.find('span', {'class': 'market-cap'}).get_text()
    float_value = card_info_divs.find('span', {'class': 'float'}).get_text()
    volume_value = card_info_divs.find('span', {'class': 'volume'}).get_text()

    # print('card_info_divs ',card_info_divs)
    print('change_value ',change_value)
    print('market_cap_value ',market_cap_value)
    print('float_value ',float_value)
    print('volume_value ',volume_value)

    news_link = base_url+news_info_link

    news_info_response = requests.get(news_link, headers=headers)

    news_info_soup = BeautifulSoup(news_info_response.content, "html.parser")

    stock_data_div = news_info_soup.find_all('div', {'class': 'article-data-panel'})[-1].find_all('div',{'class': 'news-list-item stock-data'})
    
    short_percent = None
    industry = None
    sector = None
    country = None

    for div in stock_data_div:
        print(div.find('label').get_text())
        print("==========================")
        if div.find('label').get_text() == "Short Percent":
            short_percent = div.find('span').get_text()
        
        elif div.find('label').get_text() == "Industry":
            industry = div.find('span').get_text()

        elif div.find('label').get_text() == "Sector":
            sector = div.find('span').get_text()

        elif div.find('label').get_text() == "Country":
            country = div.find('span').get_text()


    print('short_percent: ',short_percent)
    print('industry: ',industry)
    print('sector: ',sector)
    print('country: ',country)

    news_impact_star = len(news_info_soup.find('div', {'class': 'impact-container'}).find_all('div',{'class': 'full'}))
    news_sentiment_star = len(news_info_soup.find('div', {'class': 'sentiment-container'}).find_all('div',{'class': 'full'}))
    print('news_impact_star: ',news_impact_star)
    print('news_sentiment_star: ',news_sentiment_star)

    new_row = {
    'name': stock_name,
    'market_cap': market_cap_value,
    'change': change_value,
    'float': float_value,
    'volume': volume_value,
    'short_percent': short_percent,
    'industry': industry,
    'sector': sector,
    'country': country,
    'news_link': news_link,
    'news_impact_star': news_impact_star,
    'news_sentiment_star': news_sentiment_star
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv('top_gainers-stocktitan.csv')


25
I ->  0
stock_name:  ADIL
news_info_link:  /news/ADIL/
change_value  +46.00
                  %
market_cap_value  5.13M
float_value  6.48M
volume_value  5.93M
Market Cap
Float
Insiders Ownership
Institutions Ownership
Short Percent
Industry
Sector
Website
Country
City
short_percent:  10.42%
industry:  Biotechnology
sector:  Pharmaceutical Preparations
country:  United States
news_impact_star:  4
news_sentiment_star:  3
I ->  1
stock_name:  RNAZ
news_info_link:  /news/RNAZ/
change_value  +38.00
                  %
market_cap_value  13.16M
float_value  23.31M
volume_value  129.12M
Market Cap
Float
Insiders Ownership
Institutions Ownership
Short Percent
Industry
Sector
Website
Country
City
short_percent:  77.69%
industry:  Biotechnology
sector:  Pharmaceutical Preparations
country:  United States
news_impact_star:  3
news_sentiment_star:  3
I ->  2
stock_name:  SOPA
news_info_link:  /news/SOPA/
change_value  +30.00
                  %
market_cap_value  5.86M
float_value  4.4M
volume_va

In [33]:
df

,name,market_cap,change,float,volume,short_percent,industry,sector,country,news_link,news_impact_star,news_sentiment_star
0,ADIL,5.13M,+46.00\r\n %,6.48M,5.93M,10.42%,Biotechnology,Pharmaceutical Preparations,United States,https://www.stocktitan.net/news/ADIL/,4,3
1,RNAZ,13.16M,+38.00\r\n %,23.31M,129.12M,77.69%,Biotechnology,Pharmaceutical Preparations,United States,https://www.stocktitan.net/news/RNAZ/,3,3
2,SOPA,5.86M,+30.00\r\n %,4.4M,4.61M,2.62%,Software - Application,"Services-business Services, Nec",Singapore,https://www.stocktitan.net/news/SOPA/,3,3
3,KIDZ,124.77M,+28.00\r\n %,18.73M,253.49M,None,None,Services-educational Services,None,https://www.stocktitan.net/news/KIDZ/,5,3
